In [ ]:
import os
import numpy as np
import pandas as pd

import geopandas as gpd

import cmocean.cm as cmo

import xarray as xr
import rasterio
import rasterio.plot as rplt
from affine import Affine

from shapely.geometry import LineString, Point


import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable

import app

In [2]:
base_directory = r"D:\PhD\21_Experiments\TidesDamageDriver"
drainages_folder = os.path.join(
    base_directory, "02_processed", "03_drainages"
)
coastline_folder = os.path.join(
    base_directory, "01_raw", "greene2022_coastlines"
)
iceshelf_folder = os.path.join(
    base_directory, "01_raw", "greene2022_iceshelves"
)
add_path = os.path.join(
    base_directory, "01_raw", "high_res_coastline", "add_coastline_high_res_line_v7_6.shp"
)
lai_path = os.path.join(
    base_directory, "01_raw", "lai2020_vulnerability", "vulnerable.nc"
)
fuerst_path = os.path.join(
    base_directory, "01_raw", "fuerst2016_psi", "psi.nc"
)

grid_path = os.path.join(
    base_directory, "01_raw", "grid", "ne_110m_graticules_1.shp"
)
tides_folder = os.path.join(
    base_directory, "01_raw", "tides_CATS2008"
)
data_dmg_merged_folder = os.path.join(
    base_directory, "03_cleaned", "dmg_merged"
)
data_dmg_folder = os.path.join(
    base_directory, "03_cleaned", "dmg_resampled"
)
data_dmg_masked_folder = os.path.join(
    base_directory, "03_cleaned", "dmg_resampled_masked"
)
figure_folder = os.path.join(
    base_directory, "figures"
)

In [3]:
years = [2016, 2018, 2019, 2020]
xmin = 2.50e6
xmax = 2.77e6
ymax = -0.215e6
ymin = -0.591e6
vertical_line = LineString([(xmin, ymin), (xmin, ymax)])

In [4]:
all_iceshelves_files = os.listdir(iceshelf_folder)
iceshelves = [
    gpd.read_file(os.path.join(iceshelf_folder, f)).cx[xmin:xmax, ymin:ymax]
    for f in all_iceshelves_files
    if f.endswith(".shp") and any([str(s) in f for s in years])
]
ice_buffer = 150
for ice in iceshelves:
    ice.geometry = ice.buffer(ice_buffer)
    ice = ice.dissolve()
    ice.geometry = ice.buffer(-ice_buffer)
cmap = mpl.colormaps["cmo.ice"]
ice_colors = cmap(np.linspace(0.2, 0.8, 3))

buffer = 6e3
iceshelves_buffer = [i.buffer(buffer) for i in iceshelves]

In [5]:
dmgs = []
dmgs_transforms = []
dmgs_meta = []
for i_year, year in enumerate(years):
    path = os.path.join(
        data_dmg_folder, f'{year}', f'S1_{year}1101_{year}1110_3000m_dmg.tif'
    )
    with rasterio.open(path) as src:
        raster = src.read(1)
        meta = src.meta.copy()
        transform = src.transform

        raster[raster == 0] = np.nan
        masked, masked_transform = app.dmg.tiffiles.mask_dataset(
            raster, transform, iceshelves_buffer[i_year], mode="shape", filled=False
        )
        masked = masked
        masked = (masked / np.nanmax(masked)).copy()

    dmgs.append(masked)
    dmgs_transforms.append(masked_transform)
    dmgs_meta.append(meta)

In [6]:
acts = []
acts_transforms = []
acts_meta = []
for i_year, year in enumerate(years):
    path = os.path.join(
        data_dmg_folder, f'{year}', f'S1_{year}1101_{year}1110_3000m_act.tif'
    )
    with rasterio.open(path) as src:
        raster = src.read(1)
        meta = src.meta.copy()
        transform = src.transform

        raster[raster == 0] = np.nan
        masked, masked_transform = app.dmg.tiffiles.mask_dataset(
            raster, transform, iceshelves_buffer[i_year], mode="shape", filled=False
        )
        #masked = masked
        masked = (masked / np.nanmax(masked)).copy()
    acts.append(masked)
    acts_transforms.append(masked_transform)
    acts_meta.append(meta)

In [ ]:
dmg_results = []
for dmg in dmgs:
    res = app.dmg.categorize(dmg, number_bins=10, medium=5, high=7)
    dmg_results.append(res)

limit_medium = float(np.array([res["medium"]["limits"][0] for res in dmg_results]).mean())
limit_high = float(np.array([res["high"]["limits"][0] for res in dmg_results]).mean())
fractions = (float(dmg_results[0]["low"]["fraction"]), float(dmg_results[0]["medium"]["fraction"]), float(dmg_results[0]["high"]["fraction"]))

print(f"Medium limit: {limit_medium}, High limit: {limit_high}")
print(f"Fractions: {fractions}")

Medium limit: 0.02233419673441479, High limit: 0.16243124819164506
Fractions: (0.625, 0.25, 0.125)


In [9]:
act_results = []
for act in acts:
    res = app.dmg.categorize(act, number_bins=10, medium=2, high=8)
    act_results.append(res)
    #print(res["low"]["fraction"], res["medium"]["fraction"], res["high"]["fraction"])
    #print(res["limits"])
    #print(res["medium"]["limits"])

limit_medium = float(np.array([res["medium"]["limits"][0] for res in act_results]).mean())
limit_high = float(np.array([res["high"]["limits"][0] for res in act_results]).mean())
fractions = (float(act_results[0]["low"]["fraction"]), float(act_results[0]["medium"]["fraction"]), float(act_results[0]["high"]["fraction"]))

print(f"Medium limit: {limit_medium}, High limit: {limit_high}")
print(f"Fractions: {fractions}")

Medium limit: 0.02937836041674018, High limit: 0.30395886600017563
Fractions: (0.2, 0.6000000000000001, 0.19999999999999996)
